<a href="https://colab.research.google.com/github/radmm/MoSe2-structure-viewer/blob/main/MoSe2%20Viewer%20Final_505.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoSe2 Structure and Band structure
by Dev Vishwas Cse B

In [1]:
from IPython.display import HTML, display

html_code = '''
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, user-scalable=no">
<title>MoSe2 Viewer</title>
<style>
  * { margin: 0; padding: 0; box-sizing: border-box; }

  body {
    background: #0d0d1a;
    color: #e0e0e0;
    font-family: Arial, sans-serif;
    height: 600px;
    overflow: hidden;
  }

  #app { display: flex; flex-direction: column; height: 600px; }

  header {
    background: #0a0a14;
    padding: 10px 18px;
    border-bottom: 2px solid #0f3460;
    display: flex;
    align-items: center;
    justify-content: space-between;
  }

  .header-left {
    display: flex;
    align-items: baseline;
    gap: 10px;
  }

  header h1 { font-size: 1.1rem; color: #ffffff; }
  header h1 span { color: #4fc3f7; }

  .byline {
    font-size: 0.72rem;
    color: #6a8aaa;
    font-style: italic;
    letter-spacing: 0.02em;
  }

  .tabs { display: flex; gap: 8px; }
  .tab {
    padding: 6px 16px;
    border: 2px solid #0f3460;
    background: #0d0d1a;
    color: #aaa;
    font-size: 0.8rem;
    cursor: pointer;
    border-radius: 4px;
  }
  .tab.active {
    background: #0f3460;
    color: #fff;
    border-color: #4fc3f7;
  }

  .content { flex: 1; position: relative; overflow: hidden; }

  #structure-panel, #band-panel {
    position: absolute; top: 0; left: 0;
    width: 100%; height: 100%;
  }
  #structure-panel { display: block; }
  #band-panel { display: none; background: #0d0d1a; padding: 16px; }

  #three-canvas { width: 100%; height: 100%; display: block; }

  #legend {
    position: absolute;
    bottom: 16px; left: 16px;
    background: #0a0a14;
    border: 1px solid #0f3460;
    border-radius: 6px;
    padding: 10px 14px;
    font-size: 0.78rem;
  }
  .legend-item { display: flex; align-items: center; gap: 8px; margin: 3px 0; }
  .dot { width: 12px; height: 12px; border-radius: 50%; }

  #info-box {
    position: absolute;
    top: 12px; right: 12px;
    background: #0a0a14;
    border: 1px solid #0f3460;
    border-radius: 6px;
    padding: 10px 14px;
    font-size: 0.76rem;
    line-height: 1.8;
  }
  .info-label { color: #4fc3f7; font-size: 0.68rem; font-weight: bold; }

  #band-panel { flex-direction: column; gap: 12px; }
  #band-wrap {
    flex: 1;
    background: #0a0a14;
    border: 1px solid #0f3460;
    border-radius: 6px;
    overflow: hidden;
    position: relative;
  }
  #band-canvas { width: 100%; height: 100%; }

  .cards { display: flex; gap: 10px; flex-wrap: wrap; flex-shrink: 0; }
  .card {
    background: #0a0a14;
    border: 1px solid #0f3460;
    border-radius: 6px;
    padding: 10px 14px;
    flex: 1; min-width: 110px;
  }
  .card .val { font-size: 1.3rem; font-weight: bold; color: #4fc3f7; }
  .card .lbl { font-size: 0.68rem; color: #888; margin-top: 2px; }
</style>
</head>
<body>
<div id="app">
  <header>
    <div class="header-left">
      <h1>MoSe<span>2</span> Crystal Viewer</h1>
      <span class="byline">by Dev Vishwas</span>
    </div>
    <div class="tabs">
      <button class="tab active" onclick="showPanel('structure')">3D Structure</button>
      <button class="tab" onclick="showPanel('band')">Band Structure</button>
    </div>
  </header>

  <div class="content">

    <div id="structure-panel">
      <canvas id="three-canvas"></canvas>
      <div id="legend">
        <div class="legend-item"><div class="dot" style="background:#4fc3f7"></div> Mo atom</div>
        <div class="legend-item"><div class="dot" style="background:#81c784"></div> Se atom</div>
      </div>
      <div id="info-box">
        <div class="info-label">MoSe2 — 2H polymorph</div>
        <div>a = 3.288 &#8491;</div>
        <div>c = 12.927 &#8491;</div>
        <div>Mo&#8211;Se bond: 2.54 &#8491;</div>
        <div>Space group: P6&#8323;/mmc</div>
      </div>
    </div>

    <div id="band-panel" style="display:flex;">
      <div id="band-wrap"><canvas id="band-canvas"></canvas></div>
      <div class="cards">
        <div class="card"><div class="val">1.65 eV</div><div class="lbl">Direct gap (monolayer)</div></div>
        <div class="card"><div class="val">1.09 eV</div><div class="lbl">Indirect gap (bulk)</div></div>
        <div class="card"><div class="val">K point</div><div class="lbl">Direct gap location</div></div>
        <div class="card"><div class="val">Mo 4d</div><div class="lbl">VBM character</div></div>
      </div>
    </div>

  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
const canvas = document.getElementById('three-canvas');
const panel  = document.getElementById('structure-panel');

const renderer = new THREE.WebGLRenderer({ canvas, antialias: true });
renderer.setClearColor(0x0d0d1a);
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
renderer.sortObjects = true;

const scene  = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(50, 1, 0.1, 100);
camera.position.set(6, 5, 8);
camera.lookAt(0, 0, 0);

scene.add(new THREE.AmbientLight(0xffffff, 0.5));
const sun = new THREE.DirectionalLight(0xffffff, 1.0);
sun.position.set(5, 8, 5);
scene.add(sun);

const SCALE = 0.5;
const a = 3.288 * SCALE;
const c = 12.927 * SCALE;
const a1 = new THREE.Vector3(a, 0, 0);
const a2 = new THREE.Vector3(a * Math.cos(Math.PI/3), a * Math.sin(Math.PI/3), 0);

// Subtle glassy atoms
const moMat = new THREE.MeshPhongMaterial({
  color: 0x4fc3f7,
  transparent: true,
  opacity: 0.95,
  shininess: 90,
  specular: 0xaaaaaa,
});
const seMat = new THREE.MeshPhongMaterial({
  color: 0x81c784,
  transparent: true,
  opacity: 0.95,
  shininess: 90,
  specular: 0xaaaaaa,
});

// Bonds stay flat/matte
const bondMat = new THREE.MeshLambertMaterial({ color: 0x555577 });

const moGeo = new THREE.SphereGeometry(0.26, 20, 14);
const seGeo = new THREE.SphereGeometry(0.22, 20, 14);

const moPositions = [], sePositions = [];

for (let i = -1; i <= 2; i++) {
  for (let j = -1; j <= 2; j++) {
    const base = a1.clone().multiplyScalar(i).add(a2.clone().multiplyScalar(j));
    moPositions.push(base.clone());
    const dz = c/4 * 0.52, dx = a * 0.333, dy = a * 0.577;
    [
      new THREE.Vector3( dx,  dy,  dz),
      new THREE.Vector3(-dx*2, 0,  dz),
      new THREE.Vector3( dx, -dy,  dz),
      new THREE.Vector3( dx,  dy, -dz),
      new THREE.Vector3(-dx*2, 0, -dz),
      new THREE.Vector3( dx, -dy, -dz),
    ].forEach(o => sePositions.push(base.clone().add(o)));
  }
}

const group = new THREE.Group();

moPositions.forEach(p => {
  const m = new THREE.Mesh(moGeo, moMat);
  m.position.copy(p); group.add(m);
});
sePositions.forEach(p => {
  const m = new THREE.Mesh(seGeo, seMat);
  m.position.copy(p); group.add(m);
});

moPositions.forEach(mo => {
  sePositions.forEach(se => {
    const d = mo.distanceTo(se);
    if (d > 1.3) return;
    const dir = se.clone().sub(mo);
    const g = new THREE.CylinderGeometry(0.04, 0.04, d, 6);
    const m = new THREE.Mesh(g, bondMat);
    m.position.copy(mo.clone().add(se).multiplyScalar(0.5));
    m.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0), dir.normalize());
    group.add(m);
  });
});

scene.add(group);

function resize() {
  const w = panel.clientWidth, h = panel.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w / h;
  camera.updateProjectionMatrix();
}
resize();
window.addEventListener('resize', resize);

let drag = false, px = 0, py = 0, rotX = 0.3, rotY = 0, pinch = null;

canvas.addEventListener('mousedown', e => { drag = true; px = e.clientX; py = e.clientY; });
canvas.addEventListener('mousemove', e => {
  if (!drag) return;
  rotY += (e.clientX - px) * 0.01; rotX += (e.clientY - py) * 0.01;
  px = e.clientX; py = e.clientY;
});
canvas.addEventListener('mouseup', () => drag = false);
canvas.addEventListener('wheel', e => camera.position.multiplyScalar(1 + e.deltaY * 0.001));

canvas.addEventListener('touchstart', e => {
  if (e.touches.length === 1) { drag = true; px = e.touches[0].clientX; py = e.touches[0].clientY; }
  else if (e.touches.length === 2) {
    const dx = e.touches[0].clientX - e.touches[1].clientX;
    const dy = e.touches[0].clientY - e.touches[1].clientY;
    pinch = Math.sqrt(dx*dx + dy*dy);
  }
}, { passive: true });
canvas.addEventListener('touchmove', e => {
  e.preventDefault();
  if (e.touches.length === 1 && drag) {
    rotY += (e.touches[0].clientX - px) * 0.012;
    rotX += (e.touches[0].clientY - py) * 0.012;
    px = e.touches[0].clientX; py = e.touches[0].clientY;
  } else if (e.touches.length === 2 && pinch) {
    const dx = e.touches[0].clientX - e.touches[1].clientX;
    const dy = e.touches[0].clientY - e.touches[1].clientY;
    const d = Math.sqrt(dx*dx + dy*dy);
    camera.position.multiplyScalar(pinch / d);
    pinch = d;
  }
}, { passive: false });
canvas.addEventListener('touchend', () => { drag = false; pinch = null; });

function animate() {
  requestAnimationFrame(animate);
  if (!drag) rotY += 0.004;
  const r = camera.position.length();
  camera.position.x = r * Math.sin(rotY) * Math.cos(rotX);
  camera.position.y = r * Math.sin(rotX);
  camera.position.z = r * Math.cos(rotY) * Math.cos(rotX);
  camera.lookAt(0, 0, 0);
  renderer.render(scene, camera);
}
animate();

function drawBands() {
  const c2   = document.getElementById('band-canvas');
  const wrap = document.getElementById('band-wrap');
  const W = wrap.clientWidth, H = wrap.clientHeight;
  const dpr = window.devicePixelRatio || 1;
  c2.width  = W * dpr; c2.height = H * dpr;
  c2.style.width = W + 'px'; c2.style.height = H + 'px';
  const ctx = c2.getContext('2d');
  ctx.scale(dpr, dpr);

  const pad = { l: 50, r: 20, t: 20, b: 40 };
  const pw = W - pad.l - pad.r;
  const ph = H - pad.t - pad.b;

  ctx.fillStyle = '#0a0a14';
  ctx.fillRect(0, 0, W, H);

  const Emin = -4, Emax = 4;
  const ey = e => pad.t + ph * (1 - (e - Emin) / (Emax - Emin));
  const kpts   = [0, 0.3, 0.65, 1.0];
  const klabels = ['\u0393', 'M', 'K', '\u0393'];
  const kx = kpts.map(f => pad.l + f * pw);

  ctx.strokeStyle = 'rgba(255,255,255,0.06)';
  ctx.lineWidth = 1;
  for (let e = -4; e <= 4; e++) {
    ctx.beginPath(); ctx.moveTo(pad.l, ey(e)); ctx.lineTo(pad.l + pw, ey(e)); ctx.stroke();
  }

  ctx.strokeStyle = 'rgba(255,255,255,0.12)';
  ctx.setLineDash([3, 4]);
  kx.forEach(x => { ctx.beginPath(); ctx.moveTo(x, pad.t); ctx.lineTo(x, pad.t + ph); ctx.stroke(); });
  ctx.setLineDash([]);

  ctx.strokeStyle = '#e57373';
  ctx.lineWidth = 1.5;
  ctx.setLineDash([5, 4]);
  ctx.beginPath(); ctx.moveTo(pad.l, ey(0)); ctx.lineTo(pad.l + pw, ey(0)); ctx.stroke();
  ctx.setLineDash([]);
  ctx.fillStyle = '#e57373';
  ctx.font = '11px Arial';
  ctx.textAlign = 'right';
  ctx.fillText('EF', pad.l - 4, ey(0) + 4);

  const N = 200;
  function interp(kk, ee, k) {
    for (let i = 0; i < kk.length - 1; i++) {
      if (k >= kk[i] && k <= kk[i+1]) {
        const t = (k - kk[i]) / (kk[i+1] - kk[i]);
        const tc = (1 - Math.cos(t * Math.PI)) / 2;
        return ee[i] * (1 - tc) + ee[i+1] * tc;
      }
    }
    return ee[ee.length - 1];
  }

  const bands = [
    { color: '#81c784', w: 2.5, knots: [[0,-0.9],[0.3,-1.5],[0.65, 0.0],[1.0,-0.9]] },
    { color: '#81c784', w: 2,   knots: [[0,-1.8],[0.3,-2.5],[0.65,-1.2],[1.0,-1.8]] },
    { color: '#81c784', w: 2,   knots: [[0,-2.9],[0.3,-3.1],[0.65,-2.6],[1.0,-2.9]] },
    { color: '#4fc3f7', w: 2.5, knots: [[0, 1.1],[0.3, 1.45],[0.65,1.09],[1.0, 1.1]] },
    { color: '#4fc3f7', w: 2,   knots: [[0, 2.0],[0.3, 1.85],[0.65,2.2],[1.0, 2.0]] },
    { color: '#4fc3f7', w: 2,   knots: [[0, 3.1],[0.3, 2.8],[0.65,3.0],[1.0, 3.1]] },
  ];

  bands.forEach(b => {
    const kk = b.knots.map(k => k[0]);
    const ee = b.knots.map(k => k[1]);
    ctx.beginPath();
    ctx.strokeStyle = b.color;
    ctx.lineWidth = b.w;
    for (let i = 0; i < N; i++) {
      const k = i / (N - 1);
      const x = pad.l + k * pw;
      const y = ey(interp(kk, ee, k));
      i === 0 ? ctx.moveTo(x, y) : ctx.lineTo(x, y);
    }
    ctx.stroke();
  });

  const kK = kx[2];
  const yTop = ey(1.09), yBot = ey(0.0);
  ctx.fillStyle = 'rgba(255, 235, 59, 0.1)';
  ctx.fillRect(kK - 18, yTop, 36, yBot - yTop);
  ctx.strokeStyle = '#ffeb3b';
  ctx.lineWidth = 1;
  ctx.strokeRect(kK - 18, yTop, 36, yBot - yTop);
  ctx.fillStyle = '#ffeb3b';
  ctx.font = 'bold 11px Arial';
  ctx.textAlign = 'center';
  ctx.fillText('1.09 eV', kK, (yTop + yBot) / 2 + 4);

  ctx.strokeStyle = '#aaa';
  ctx.lineWidth = 1.5;
  ctx.beginPath(); ctx.moveTo(pad.l, pad.t); ctx.lineTo(pad.l, pad.t + ph); ctx.stroke();
  ctx.beginPath(); ctx.moveTo(pad.l, pad.t + ph); ctx.lineTo(pad.l + pw, pad.t + ph); ctx.stroke();

  ctx.fillStyle = '#aaa';
  ctx.font = '11px Arial';
  ctx.textAlign = 'right';
  for (let e = -4; e <= 4; e += 2) ctx.fillText(e, pad.l - 6, ey(e) + 4);

  ctx.save();
  ctx.translate(13, pad.t + ph / 2);
  ctx.rotate(-Math.PI / 2);
  ctx.textAlign = 'center';
  ctx.fillStyle = '#ccc';
  ctx.font = '12px Arial';
  ctx.fillText('Energy (eV)', 0, 0);
  ctx.restore();

  ctx.textAlign = 'center';
  ctx.fillStyle = '#ccc';
  ctx.font = 'bold 13px Arial';
  klabels.forEach((l, i) => ctx.fillText(l, kx[i], pad.t + ph + 26));

  ctx.font = '11px Arial';
  ctx.textAlign = 'left';
  ctx.fillStyle = '#81c784'; ctx.fillText('Valence bands',    pad.l + 6, pad.t + 14);
  ctx.fillStyle = '#4fc3f7'; ctx.fillText('Conduction bands', pad.l + 6, pad.t + 28);
}

function showPanel(name) {
  document.querySelectorAll('.tab').forEach((t, i) =>
    t.classList.toggle('active', (i===0 && name==='structure') || (i===1 && name==='band'))
  );
  document.getElementById('structure-panel').style.display = name === 'structure' ? 'block' : 'none';
  document.getElementById('band-panel').style.display     = name === 'band'      ? 'flex'  : 'none';
  if (name === 'structure') resize();
  else setTimeout(drawBands, 40);
}
window.showPanel = showPanel;
window.addEventListener('resize', () => {
  resize();
  if (document.getElementById('band-panel').style.display !== 'none') drawBands();
});
</script>
</body>
</html>
'''

display(HTML(f'<div style="height:620px">{html_code}</div>'))

In [ ]:
# Save and download
with open('MoSe2_Viewer.html', 'w') as f:
    f.write(html_code)

try:
    from google.colab import files
    files.download('MoSe2_Viewer.html')
    print('Download started!')
except ImportError:
    print('Saved as MoSe2_Viewer.html')